# Talea LST webapp data pipeline

This notebook rebuilds the Talea land-surface-temperature webapp data products from raw Earth Engine exports and vector boundary files. It is designed to be the only source file needed to rerun the logic.

Each section explains what it reads, what it writes, and what to check before running it.

The pipeline has two parts:

1. Earth Engine export preparation: the notebook generates JavaScript snippets that you paste into the Earth Engine Code Editor. Those scripts export raw rasters to Google Drive.
2. Local processing: the notebook reads those downloaded rasters and produces yearly LST layers, climatology, anomalies, hotspot masks, composite heat indices, webapp rasters, vector summaries, and optional clipped rasters.

Typical use:

1. Confirm or edit `CFG` in the configuration section.
2. Generate the Earth Engine scripts and run them manually in the Earth Engine Code Editor.
3. Download the exported files into the raw folders checked by the notebook.
4. Run the local pipeline section.
5. Read the validation table at the end before pushing or sharing outputs.

Required Python packages: `numpy`, `pandas`, `rasterio`, `matplotlib`, `requests`, `geopandas`, and `rasterstats`.


## Imports

This cell loads the Python libraries used by the rest of the notebook.

What these libraries do:

- `numpy` and `pandas`: numeric arrays and CSV tables.
- `rasterio`: read, write, reproject, and clip GeoTIFF rasters.
- `matplotlib`: create the albedo versus DeltaLST scatter plot.
- `geopandas`: read and write GeoJSON/GPKG vector files.
- `rasterstats`: compute per-polygon raster statistics.
- `requests`: optionally download vector files from open-data URLs.

The vector-related libraries are imported with fallbacks. This means raster-only steps can still be inspected even if vector packages are missing, but steps `06A`, `06B`, and `07` require the vector dependencies to be installed.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field
from pathlib import Path
from string import Template
import json
import math
import re
import textwrap
import time
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from rasterio.enums import Resampling
from rasterio.mask import mask as rasterio_mask
from rasterio.warp import reproject

try:
    import geopandas as gpd
except ImportError:
    gpd = None

try:
    import requests
except ImportError:
    requests = None

try:
    from rasterstats import zonal_stats
except ImportError:
    zonal_stats = None

try:
    from IPython.display import Code, Markdown, display
except ImportError:
    Code = Markdown = display = None

warnings.filterwarnings("ignore", category=RuntimeWarning, message="Mean of empty slice")
warnings.filterwarnings("ignore", category=RuntimeWarning, message="Degrees of freedom")


## Configuration

This is the main section a user should edit for a new city or dataset.

The `PipelineConfig` object defines:

- the city name used in raster filenames;
- the bounding box and projected CRS used by Earth Engine exports;
- the LST year range and webapp year;
- the input/output folder layout;
- hotspot thresholds and nodata values;
- optional vector download URLs.

For this repository, `CFG` is currently set to `Cluj-Napoca` and `data_folder="data_cluj_napoca"`, because that is where the active raw inputs are stored. If you run another municipality, update the preset, `data_folder`, CRS, bounding box, Drive folder names, and vector filenames as needed.

Important conventions:

- `city` is used as the raster filename prefix, for example `Cluj-Napoca_LST_2025_summer_median_30m.tif`.
- `city_slug` is used for vector filenames, for example `cluj-napoca_boundary.geojson`.
- `years_lst` controls how many bands the LST stack must contain.
- `webapp_year` controls which year is used for NDVI, albedo, DeltaLST, webapp rasters, and polygon summaries.

After this cell runs, all expected folders are created if they do not already exist.


In [ ]:
CITY_PRESETS = {
    "Bologna": {
        "city": "Bologna",
        "roi_bbox": (11.220132616970325, 44.41434419520011, 11.441381077496928, 44.56324809522889),
        "default_crs": "EPSG:32632",
    },
    "Cluj-Napoca": {
        "city": "Cluj-Napoca",
        "roi_bbox": (23.42, 46.69, 23.72, 46.85),
        "default_crs": "EPSG:32634",
    },
    "Marseille": {
        "city": "Marseille",
        "roi_bbox": (5.24, 43.18, 5.58, 43.42),
        "default_crs": "EPSG:32631",
    },
    "Riga": {
        "city": "Riga",
        "roi_bbox": (23.90, 56.83, 24.35, 57.12),
        "default_crs": "EPSG:32635",
    },
}


def default_project_root() -> Path:
    cwd = Path.cwd().resolve()
    return cwd.parent if cwd.name.lower() == "notebooks" else cwd


def default_city_slug(city: str) -> str:
    return city.lower().replace(" ", "_")


@dataclass
class PipelineConfig:
    city: str
    roi_bbox: tuple[float, float, float, float]
    default_crs: str
    years_lst: tuple[int, ...] = field(default_factory=lambda: tuple(range(2013, 2026)))
    years_ndvi: tuple[int, ...] = field(default_factory=lambda: tuple(range(2017, 2026)))
    webapp_year: int = 2025
    project_root: Path = field(default_factory=default_project_root)
    data_folder: str = "data"
    city_slug: str | None = None

    min_valid_years: int = 3
    hotspot_z_temp_threshold: float = 1.0
    hotspot_z_temp_strong_threshold: float = 1.5
    hotspot_z_spatial_threshold: float = 1.0
    hotspot_percentile: float = 95.0
    hotspot_persistence_threshold: int = 3
    default_nodata: float = -9999.0

    drive_folder_lst: str = "GEE_LST"
    drive_folder_ndvi: str = "GEE_NDVI"
    drive_folder_modis: str = "GEE_MODIS"
    drive_folder_albedo: str = "GEE_ALBEDO"

    vector_download_urls: dict[str, str] = field(default_factory=dict)
    boundary_source_filename: str | None = None

    def __post_init__(self) -> None:
        self.project_root = Path(self.project_root).expanduser().resolve()
        self.roi_bbox = tuple(float(v) for v in self.roi_bbox)
        self.years_lst = tuple(int(y) for y in self.years_lst)
        self.years_ndvi = tuple(int(y) for y in self.years_ndvi)
        if self.city_slug is None:
            self.city_slug = default_city_slug(self.city)
        if self.boundary_source_filename is None:
            self.boundary_source_filename = f"{self.city_slug}_statistical_areas.geojson"

    @classmethod
    def from_preset(cls, preset_name: str, **overrides) -> "PipelineConfig":
        values = dict(CITY_PRESETS[preset_name])
        values.update(overrides)
        return cls(**values)

    @property
    def period_label(self) -> str:
        return f"{min(self.years_lst)}_{max(self.years_lst)}"

    @property
    def data_dir(self) -> Path:
        return self.project_root / self.data_folder

    @property
    def raw_dir(self) -> Path:
        return self.data_dir / "raw"

    @property
    def intermediate_dir(self) -> Path:
        return self.data_dir / "intermediate"

    @property
    def output_dir(self) -> Path:
        return self.data_dir / "outputs"

    @property
    def vector_dir(self) -> Path:
        return self.data_dir / "vector"

    @property
    def csv_data_download_dir(self) -> Path:
        return self.data_dir / "csv_data_download"

    @property
    def raw_bands_dir(self) -> Path:
        return self.raw_dir / "bands"

    @property
    def raw_lst_dir(self) -> Path:
        return self.raw_dir / "gee_lst"

    @property
    def raw_ndvi_dir(self) -> Path:
        return self.raw_dir / "gee_ndvi"

    @property
    def raw_albedo_dir(self) -> Path:
        return self.raw_dir / "gee_albedo"

    @property
    def raw_modis_dir(self) -> Path:
        return self.raw_dir / "gee_modis"

    @property
    def temporal_dir(self) -> Path:
        return self.intermediate_dir / "temporal"

    @property
    def spatial_dir(self) -> Path:
        return self.intermediate_dir / "spatial"

    @property
    def normalized_dir(self) -> Path:
        return self.intermediate_dir / "normalized"

    @property
    def composites_dir(self) -> Path:
        return self.intermediate_dir / "composites"

    @property
    def hotspots_dir(self) -> Path:
        return self.intermediate_dir / "hotspots"

    @property
    def webapp_rasters_dir(self) -> Path:
        return self.output_dir / "webapp_rasters"

    @property
    def webapp_vectors_dir(self) -> Path:
        return self.output_dir / "webapp_vectors"

    @property
    def summaries_dir(self) -> Path:
        return self.output_dir / "summaries"

    @property
    def metadata_dir(self) -> Path:
        return self.output_dir / "metadata"

    @property
    def boundary_file(self) -> Path:
        return self.vector_dir / f"{self.city_slug}_boundary.geojson"

    @property
    def districts_file(self) -> Path:
        return self.vector_dir / f"{self.city_slug}_statistical_areas.geojson"

    @property
    def lst_template(self) -> str:
        return f"{self.city}_LST_{{year}}_summer_median_30m.tif"

    @property
    def lst_mean_template(self) -> str:
        return f"{self.city}_LST_{{year}}_summer_mean_30m.tif"

    @property
    def lst_validobs_template(self) -> str:
        return f"{self.city}_LST_{{year}}_summer_validObsCount_30m.tif"

    @property
    def lst_stack_median_file(self) -> str:
        return f"{self.city}_LST_{self.period_label}_summer_median_stack_30m.tif"

    @property
    def lst_stack_mean_file(self) -> str:
        return f"{self.city}_LST_{self.period_label}_summer_mean_stack_30m.tif"

    @property
    def lst_stack_validobs_file(self) -> str:
        return f"{self.city}_LST_{self.period_label}_summer_validObs_stack_30m.tif"

    @property
    def ndvi_file(self) -> str:
        return f"{self.city}_NDVI_{self.webapp_year}_summer_JJA_median.tif"

    @property
    def albedo_file(self) -> str:
        return f"{self.city}_Albedo_{self.webapp_year}_summer_JJA_mean_30m.tif"

    @property
    def deltalst_file(self) -> str:
        return f"{self.city}_MODIS_DeltaLST_{self.webapp_year}_summer_JJA_mean_1km.tif"

    @property
    def lst_metadata_file(self) -> str:
        return f"{self.city}_LST_metadata_{self.period_label}_v2.csv"

    def ensure_directories(self) -> None:
        for path in [
            self.data_dir, self.raw_dir, self.intermediate_dir, self.output_dir,
            self.vector_dir, self.csv_data_download_dir,
            self.raw_bands_dir, self.raw_lst_dir, self.raw_ndvi_dir,
            self.raw_albedo_dir, self.raw_modis_dir,
            self.temporal_dir, self.spatial_dir, self.normalized_dir,
            self.composites_dir, self.hotspots_dir,
            self.webapp_rasters_dir, self.webapp_vectors_dir,
            self.summaries_dir, self.metadata_dir,
        ]:
            path.mkdir(parents=True, exist_ok=True)

    def as_dict(self) -> dict:
        return {
            "city": self.city,
            "city_slug": self.city_slug,
            "roi_bbox": self.roi_bbox,
            "default_crs": self.default_crs,
            "years_lst": f"{min(self.years_lst)}-{max(self.years_lst)}",
            "webapp_year": self.webapp_year,
            "project_root": str(self.project_root),
            "data_dir": str(self.data_dir),
        }


# Edit this block for a new municipality.
CFG = PipelineConfig.from_preset(
    "Cluj-Napoca",
    # project_root=Path("/absolute/path/to/project/root"),
    data_folder="data_cluj_napoca",
    # drive_folder_lst="GEE_LST_MY_CITY",
    # drive_folder_ndvi="GEE_NDVI_MY_CITY",
    # drive_folder_modis="GEE_MODIS_MY_CITY",
    # drive_folder_albedo="GEE_ALBEDO_MY_CITY",
    # vector_download_urls={
    #     "my_city_statistical_areas.geojson": "https://example.org/districts.geojson",
    # },
    # boundary_source_filename="my_city_statistical_areas.geojson",
)

CFG.ensure_directories()
pd.DataFrame([CFG.as_dict()]).T.rename(columns={0: "value"})


## Step 00 - Earth Engine export scripts

This section generates the JavaScript code that must be run in the Google Earth Engine Code Editor. The Python notebook does not run Earth Engine exports directly, because Earth Engine requires authentication and each export task must usually be started from the Code Editor task panel.

The generated scripts produce four raw input groups:

- LST multi-band stacks from Landsat 8/9 for all years in `years_lst`.
- NDVI for `webapp_year` from Sentinel-2.
- Day-night DeltaLST for `webapp_year` from MODIS.
- A Landsat metadata CSV with scene date, satellite, and valid-pixel percentage.

What the user should do:

1. Run this cell.
2. Copy each displayed JavaScript block into https://code.earthengine.google.com.
3. Start each Earth Engine export task.
4. Download the exported files from Google Drive.
5. Place them in the folders listed by the next raw input check.

The local processing steps cannot recreate these raw satellite exports; they only process them after download.


In [ ]:
def js_literal(value) -> str:
    return json.dumps(value)


def render_js(template: str, **values) -> str:
    rendered_values = {key: js_literal(value) for key, value in values.items()}
    return Template(textwrap.dedent(template).strip()).substitute(rendered_values) + "\n"


def gee_lst_stack_script(cfg: PipelineConfig) -> str:
    return render_js(
        r'''
        // LST STACK EXPORT (Landsat 8 + 9, summer JJA)
        // Produces median, mean, and valid-observation multi-band stacks.
        // Run in https://code.earthengine.google.com

        var CITY = $city;
        var BBOX = $bbox;
        var YEAR_START = $year_start;
        var YEAR_END = $year_end;
        var CRS = $crs;
        var DRIVE_FOLDER = $drive_folder;

        var roi = ee.Geometry.Rectangle(BBOX, 'EPSG:4326', false);
        var years = ee.List.sequence(YEAR_START, YEAR_END);
        var periodLabel = YEAR_START + '_' + YEAR_END;

        var l8 = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2');
        var l9 = ee.ImageCollection('LANDSAT/LC09/C02/T1_L2');

        function maskLandsatLST(image) {
          var qa = image.select('QA_PIXEL');
          var mask = qa.bitwiseAnd(1 << 1).eq(0)
            .and(qa.bitwiseAnd(1 << 2).eq(0))
            .and(qa.bitwiseAnd(1 << 3).eq(0))
            .and(qa.bitwiseAnd(1 << 4).eq(0))
            .and(qa.bitwiseAnd(1 << 5).eq(0));

          var lst = image.select('ST_B10')
            .multiply(0.00341802)
            .add(149.0)
            .subtract(273.15)
            .rename('LST');

          var valid = ee.Image.constant(1).updateMask(mask).rename('valid');

          return ee.Image.cat([lst.updateMask(mask), valid])
            .copyProperties(image, ['system:time_start', 'SPACECRAFT_ID', 'LANDSAT_PRODUCT_ID']);
        }

        function buildYearImage(year) {
          year = ee.Number(year).toInt();
          var yearStr = year.format('%.0f');
          var start = ee.Date.fromYMD(year, 6, 1);
          var end = ee.Date.fromYMD(year, 9, 1);

          var filter = ee.Filter.and(
            ee.Filter.bounds(roi),
            ee.Filter.date(start, end),
            ee.Filter.eq('PROCESSING_LEVEL', 'L2SP')
          );

          var col = l8.filter(filter).merge(l9.filter(filter)).map(maskLandsatLST);
          var lstCol = col.select('LST');
          var validCol = col.select('valid');

          var median = lstCol.median().clip(roi).rename(ee.String('LST_').cat(yearStr));
          var mean = lstCol.mean().clip(roi).rename(ee.String('LST_').cat(yearStr));
          var validObs = validCol.sum().clip(roi).rename(ee.String('validObs_').cat(yearStr));

          return ee.Dictionary({year: year, median: median, mean: mean, validObs: validObs});
        }

        var yearDicts = years.map(buildYearImage);

        var medianStack = ee.ImageCollection.fromImages(
          yearDicts.map(function(d) { return ee.Image(ee.Dictionary(d).get('median')); })
        ).toBands();

        var meanStack = ee.ImageCollection.fromImages(
          yearDicts.map(function(d) { return ee.Image(ee.Dictionary(d).get('mean')); })
        ).toBands();

        var validObsStack = ee.ImageCollection.fromImages(
          yearDicts.map(function(d) { return ee.Image(ee.Dictionary(d).get('validObs')); })
        ).toBands();

        var medianBandNames = years.map(function(y) { return ee.String('LST_').cat(ee.Number(y).toInt().format('%.0f')); });
        var meanBandNames = years.map(function(y) { return ee.String('LST_').cat(ee.Number(y).toInt().format('%.0f')); });
        var validBandNames = years.map(function(y) { return ee.String('validObs_').cat(ee.Number(y).toInt().format('%.0f')); });

        medianStack = medianStack.rename(medianBandNames);
        meanStack = meanStack.rename(meanBandNames);
        validObsStack = validObsStack.rename(validBandNames);

        var medianName = CITY + '_LST_' + periodLabel + '_summer_median_stack_30m';
        var meanName = CITY + '_LST_' + periodLabel + '_summer_mean_stack_30m';
        var validName = CITY + '_LST_' + periodLabel + '_summer_validObs_stack_30m';

        Export.image.toDrive({
          image: medianStack.toFloat().unmask(-9999),
          description: medianName,
          folder: DRIVE_FOLDER,
          fileNamePrefix: medianName,
          region: roi, scale: 30, crs: CRS, maxPixels: 1e13,
          formatOptions: {noData: -9999}
        });

        Export.image.toDrive({
          image: meanStack.toFloat().unmask(-9999),
          description: meanName,
          folder: DRIVE_FOLDER,
          fileNamePrefix: meanName,
          region: roi, scale: 30, crs: CRS, maxPixels: 1e13,
          formatOptions: {noData: -9999}
        });

        Export.image.toDrive({
          image: validObsStack.toFloat().unmask(-9999),
          description: validName,
          folder: DRIVE_FOLDER,
          fileNamePrefix: validName,
          region: roi, scale: 30, crs: CRS, maxPixels: 1e13,
          formatOptions: {noData: -9999}
        });
        ''',
        city=cfg.city,
        bbox=list(cfg.roi_bbox),
        year_start=min(cfg.years_lst),
        year_end=max(cfg.years_lst),
        crs=cfg.default_crs,
        drive_folder=cfg.drive_folder_lst,
    )


def gee_ndvi_deltalst_script(cfg: PipelineConfig) -> str:
    return render_js(
        r'''
        // NDVI (Sentinel-2) and Day-Night DeltaLST (MODIS) export.
        // Run in https://code.earthengine.google.com

        var CITY = $city;
        var BBOX = $bbox;
        var WEBAPP_YEAR = $webapp_year;
        var CRS = $crs;
        var DRIVE_FOLDER_NDVI = $drive_folder_ndvi;
        var DRIVE_FOLDER_MODIS = $drive_folder_modis;

        var roi = ee.Geometry.Rectangle(BBOX, 'EPSG:4326', false);
        var start = WEBAPP_YEAR + '-06-01';
        var end = WEBAPP_YEAR + '-09-01';

        var s2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED');
        var cloud = ee.ImageCollection('GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED');
        var modis = ee.ImageCollection('MODIS/061/MOD11A2');

        function maskS2(image) {
          var clear = image.select('cs').gte(0.6);
          return image.select(['B2','B3','B4','B8','B11','B12'])
            .updateMask(clear).multiply(0.0001)
            .copyProperties(image, ['system:time_start']);
        }

        var s2Image = s2.filterBounds(roi).filterDate(start, end)
          .linkCollection(cloud.filterBounds(roi).filterDate(start, end), ['cs'])
          .map(maskS2).median().clip(roi);

        var ndvi = s2Image.normalizedDifference(['B8','B4']).rename('NDVI');
        var ndviName = CITY + '_NDVI_' + WEBAPP_YEAR + '_summer_JJA_median';

        Export.image.toDrive({
          image: ndvi.toFloat().unmask(-9999),
          description: ndviName,
          folder: DRIVE_FOLDER_NDVI,
          fileNamePrefix: ndviName,
          region: roi, scale: 10, crs: CRS, maxPixels: 1e13,
          formatOptions: {noData: -9999}
        });

        var modisFiltered = modis.filterBounds(roi).filterDate(start, end);
        var day = modisFiltered.select('LST_Day_1km').mean().multiply(0.02).subtract(273.15);
        var night = modisFiltered.select('LST_Night_1km').mean().multiply(0.02).subtract(273.15);
        var delta = day.subtract(night).rename('DeltaLST').clip(roi);
        var deltaName = CITY + '_MODIS_DeltaLST_' + WEBAPP_YEAR + '_summer_JJA_mean_1km';

        Export.image.toDrive({
          image: delta.toFloat().unmask(-9999),
          description: deltaName,
          folder: DRIVE_FOLDER_MODIS,
          fileNamePrefix: deltaName,
          region: roi, scale: 1000, crs: CRS, maxPixels: 1e13,
          formatOptions: {noData: -9999}
        });
        ''',
        city=cfg.city,
        bbox=list(cfg.roi_bbox),
        webapp_year=cfg.webapp_year,
        crs=cfg.default_crs,
        drive_folder_ndvi=cfg.drive_folder_ndvi,
        drive_folder_modis=cfg.drive_folder_modis,
    )


def gee_albedo_script(cfg: PipelineConfig) -> str:
    return render_js(
        r'''
        // Albedo (Landsat 8 + 9, summer JJA) export.
        // Run in https://code.earthengine.google.com

        var CITY = $city;
        var BBOX = $bbox;
        var WEBAPP_YEAR = $webapp_year;
        var CRS = $crs;
        var DRIVE_FOLDER = $drive_folder;

        var roi = ee.Geometry.Rectangle(BBOX, 'EPSG:4326', false);
        var start = WEBAPP_YEAR + '-06-01';
        var end = WEBAPP_YEAR + '-09-01';

        var l8 = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2');
        var l9 = ee.ImageCollection('LANDSAT/LC09/C02/T1_L2');

        function maskAndScaleSR(image) {
          var qa = image.select('QA_PIXEL');
          var mask = qa.bitwiseAnd(1 << 1).eq(0)
            .and(qa.bitwiseAnd(1 << 2).eq(0))
            .and(qa.bitwiseAnd(1 << 3).eq(0))
            .and(qa.bitwiseAnd(1 << 4).eq(0))
            .and(qa.bitwiseAnd(1 << 5).eq(0));

          var sr = image.select(['SR_B2','SR_B4','SR_B5','SR_B6','SR_B7'])
            .multiply(0.0000275).add(-0.2).updateMask(mask);

          var albedo = sr.expression(
            '0.356 * blue + 0.130 * red + 0.373 * nir + 0.085 * swir1 + 0.072 * swir2 - 0.0018', {
              blue: sr.select('SR_B2'),
              red: sr.select('SR_B4'),
              nir: sr.select('SR_B5'),
              swir1: sr.select('SR_B6'),
              swir2: sr.select('SR_B7')
            }
          ).rename('Albedo');

          return albedo.copyProperties(image, ['system:time_start']);
        }

        var filter = ee.Filter.and(
          ee.Filter.bounds(roi),
          ee.Filter.date(start, end),
          ee.Filter.eq('PROCESSING_LEVEL', 'L2SP')
        );

        var albedo = l8.filter(filter).merge(l9.filter(filter))
          .map(maskAndScaleSR).mean().clip(roi);

        var albName = CITY + '_Albedo_' + WEBAPP_YEAR + '_summer_JJA_mean_30m';

        Export.image.toDrive({
          image: albedo.toFloat().unmask(-9999),
          description: albName,
          folder: DRIVE_FOLDER,
          fileNamePrefix: albName,
          region: roi, scale: 30, crs: CRS, maxPixels: 1e13,
          formatOptions: {noData: -9999}
        });
        ''',
        city=cfg.city,
        bbox=list(cfg.roi_bbox),
        webapp_year=cfg.webapp_year,
        crs=cfg.default_crs,
        drive_folder=cfg.drive_folder_albedo,
    )


def gee_lst_metadata_script(cfg: PipelineConfig) -> str:
    return render_js(
        r'''
        // Per-scene Landsat 8/9 metadata export for the LST summer filter.
        // Produces date, time_UTC, satellite, valid_pixels_percent, landsat_id.
        // Run in https://code.earthengine.google.com

        var CITY = $city;
        var BBOX = $bbox;
        var YEAR_START = $year_start;
        var YEAR_END = $year_end;
        var DRIVE_FOLDER = $drive_folder;

        var roi = ee.Geometry.Rectangle(BBOX, 'EPSG:4326', false);
        var periodLabel = YEAR_START + '_' + YEAR_END;

        var l8 = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2');
        var l9 = ee.ImageCollection('LANDSAT/LC09/C02/T1_L2');

        function summerFilter(year) {
          var start = ee.Date.fromYMD(year, 6, 1);
          var end = ee.Date.fromYMD(year, 9, 1);
          return ee.Filter.and(
            ee.Filter.bounds(roi),
            ee.Filter.date(start, end),
            ee.Filter.eq('PROCESSING_LEVEL', 'L2SP')
          );
        }

        function validPixelPercent(image) {
          var qa = image.select('QA_PIXEL');
          var mask = qa.bitwiseAnd(1 << 1).eq(0)
            .and(qa.bitwiseAnd(1 << 2).eq(0))
            .and(qa.bitwiseAnd(1 << 3).eq(0))
            .and(qa.bitwiseAnd(1 << 4).eq(0))
            .and(qa.bitwiseAnd(1 << 5).eq(0));

          var validImg = ee.Image.constant(1).updateMask(mask).rename('valid');
          var totalImg = ee.Image.constant(1).rename('tot');

          var counts = validImg.addBands(totalImg).reduceRegion({
            reducer: ee.Reducer.sum(),
            geometry: roi,
            scale: 30,
            maxPixels: 1e13,
            bestEffort: true
          });

          var validCount = ee.Number(counts.get('valid'));
          var totalCount = ee.Number(counts.get('tot'));
          return ee.Algorithms.If(
            totalCount.gt(0),
            validCount.divide(totalCount).multiply(100),
            null
          );
        }

        function buildRow(image) {
          var ts = ee.Date(image.get('system:time_start'));
          return ee.Feature(null, {
            date: ts.format('YYYY-MM-dd'),
            time_UTC: ts.format('HH:mm:ss'),
            satellite: image.get('SPACECRAFT_ID'),
            valid_pixels_percent: validPixelPercent(image),
            landsat_id: image.get('LANDSAT_PRODUCT_ID')
          });
        }

        var years = ee.List.sequence(YEAR_START, YEAR_END);

        var allFeatures = ee.FeatureCollection(years.iterate(function(year, prev) {
          year = ee.Number(year);
          var col = l8.filter(summerFilter(year)).merge(l9.filter(summerFilter(year)));
          return ee.FeatureCollection(prev).merge(col.map(buildRow));
        }, ee.FeatureCollection([])));

        var sorted = allFeatures.sort('date');
        var outName = CITY + '_LST_metadata_' + periodLabel + '_v2';

        Export.table.toDrive({
          collection: sorted,
          description: outName,
          folder: DRIVE_FOLDER,
          fileNamePrefix: outName,
          fileFormat: 'CSV',
          selectors: ['date', 'time_UTC', 'satellite', 'valid_pixels_percent', 'landsat_id']
        });
        ''',
        city=cfg.city,
        bbox=list(cfg.roi_bbox),
        year_start=min(cfg.years_lst),
        year_end=max(cfg.years_lst),
        drive_folder=cfg.drive_folder_lst,
    )


def build_gee_scripts(cfg: PipelineConfig) -> dict[str, str]:
    return {
        "00a_export_lst_stack.js": gee_lst_stack_script(cfg),
        "00b_export_ndvi_deltalst.js": gee_ndvi_deltalst_script(cfg),
        "00c_export_albedo_landsat.js": gee_albedo_script(cfg),
        "00d_export_lst_metadata.js": gee_lst_metadata_script(cfg),
    }


SHOW_GEE_SCRIPTS = True
if SHOW_GEE_SCRIPTS and display is not None:
    for filename, script in build_gee_scripts(CFG).items():
        display(Markdown(f"### `{filename}`"))
        display(Code(script, language="javascript"))
elif SHOW_GEE_SCRIPTS:
    for filename, script in build_gee_scripts(CFG).items():
        print(f"\n===== {filename} =====\n{script}")


## Raw input checks

This section checks whether the raw files expected by the local pipeline are present.

The most important files are:

- three LST stack GeoTIFFs in `raw/bands`;
- one NDVI GeoTIFF in `raw/gee_ndvi`;
- one albedo GeoTIFF in `raw/gee_albedo`;
- one MODIS DeltaLST GeoTIFF in `raw/gee_modis`.

The LST metadata CSV is also checked, but it is not required by the local raster pipeline. It is useful as documentation of the Landsat scenes used by Earth Engine.

If one of the required rasters is missing, stop here and place the downloaded Earth Engine output in the path shown by the table before running later steps.


In [ ]:
def expected_raw_inputs(cfg: PipelineConfig) -> dict[str, Path]:
    return {
        "LST median stack": cfg.raw_bands_dir / cfg.lst_stack_median_file,
        "LST mean stack": cfg.raw_bands_dir / cfg.lst_stack_mean_file,
        "LST validObs stack": cfg.raw_bands_dir / cfg.lst_stack_validobs_file,
        "NDVI raster": cfg.raw_ndvi_dir / cfg.ndvi_file,
        "Albedo raster": cfg.raw_albedo_dir / cfg.albedo_file,
        "MODIS DeltaLST raster": cfg.raw_modis_dir / cfg.deltalst_file,
        "LST metadata CSV": cfg.csv_data_download_dir / cfg.lst_metadata_file,
    }


def check_paths(paths: dict[str, Path]) -> pd.DataFrame:
    rows = []
    for label, path in paths.items():
        rows.append({
            "input": label,
            "exists": path.exists(),
            "path": str(path),
        })
    return pd.DataFrame(rows)


check_paths(expected_raw_inputs(CFG))


## Raster utilities

These helper functions are the reusable raster toolkit used by the pipeline.

They handle common geospatial tasks that appear in multiple steps:

- converting nodata values such as `-9999` to `NaN` for calculations;
- reading and writing single-band GeoTIFFs with consistent metadata;
- calculating summary statistics for CSV reports;
- min-max normalizing rasters to the range `[0, 1]`;
- reprojecting one raster onto another raster grid;
- checking that several rasters share the same grid before combining them.

If a later step fails with a grid or CRS error, this section is where that consistency check is implemented.


In [ ]:
DEFAULT_FILL_NODATA = -9999.0


def require_file(path: Path, label: str | None = None) -> Path:
    path = Path(path)
    if not path.exists():
        prefix = f"{label}: " if label else ""
        raise FileNotFoundError(f"{prefix}{path}")
    return path


def mask_nodata_values(arr, nodata=None, extra_nodata=(DEFAULT_FILL_NODATA,)):
    data = arr.astype("float32", copy=True)

    nodata_values = []
    if nodata is not None and np.isfinite(nodata):
        nodata_values.append(float(nodata))
    nodata_values.extend(float(v) for v in extra_nodata if v is not None and np.isfinite(v))

    for value in nodata_values:
        data[data == value] = np.nan
    return data


def read_raster(path: Path):
    with rasterio.open(path) as src:
        arr = src.read(1).astype("float32")
        profile = src.profile.copy()
        nodata = src.nodata
    arr = mask_nodata_values(arr, nodata)
    return arr, profile


def write_raster(path: Path, arr, profile, nodata=-9999.0):
    out = profile.copy()
    out.update(dtype="float32", count=1, compress="deflate", nodata=nodata)
    data = np.where(np.isfinite(arr), arr.astype("float32"), nodata).astype("float32")
    path.parent.mkdir(parents=True, exist_ok=True)
    with rasterio.open(path, "w", **out) as dst:
        dst.write(data, 1)


def raster_stats(arr):
    vals = arr[np.isfinite(arr)]
    if vals.size == 0:
        return {
            "count": 0, "mean": np.nan, "std": np.nan, "min": np.nan,
            "p05": np.nan, "p25": np.nan, "p50": np.nan, "p75": np.nan,
            "p95": np.nan, "max": np.nan,
        }
    return {
        "count": int(vals.size),
        "mean": float(np.nanmean(vals)),
        "std": float(np.nanstd(vals)),
        "min": float(np.nanmin(vals)),
        "p05": float(np.nanpercentile(vals, 5)),
        "p25": float(np.nanpercentile(vals, 25)),
        "p50": float(np.nanpercentile(vals, 50)),
        "p75": float(np.nanpercentile(vals, 75)),
        "p95": float(np.nanpercentile(vals, 95)),
        "max": float(np.nanmax(vals)),
    }


def minmax_normalize(arr, mask=None):
    data = arr.copy().astype("float32")
    vals = data[np.isfinite(data)] if mask is None else data[np.isfinite(data) & mask]
    if vals.size == 0:
        return np.full_like(data, np.nan, dtype="float32")
    vmin, vmax = np.nanmin(vals), np.nanmax(vals)
    if np.isclose(vmin, vmax):
        out = np.full_like(data, np.nan, dtype="float32")
        out[np.isfinite(data)] = 0.0
        return out
    out = (data - vmin) / (vmax - vmin)
    out[~np.isfinite(data)] = np.nan
    return out.astype("float32")


def reproject_to_match(src_arr, src_profile, dst_profile, resampling=Resampling.bilinear):
    dst = np.full((dst_profile["height"], dst_profile["width"]), np.nan, dtype="float32")
    src_nodata = src_profile.get("nodata", None)
    src_fill = src_nodata if src_nodata is not None else DEFAULT_FILL_NODATA
    reproject(
        source=np.where(np.isfinite(src_arr), src_arr, src_fill),
        destination=dst,
        src_transform=src_profile["transform"],
        src_crs=src_profile["crs"],
        src_nodata=src_fill,
        dst_transform=dst_profile["transform"],
        dst_crs=dst_profile["crs"],
        dst_nodata=DEFAULT_FILL_NODATA,
        resampling=resampling,
    )
    dst[dst == DEFAULT_FILL_NODATA] = np.nan
    return dst


def ensure_same_grid(arrays_profiles):
    first = arrays_profiles[0][1]
    for _, profile in arrays_profiles[1:]:
        same = (
            profile["width"] == first["width"]
            and profile["height"] == first["height"]
            and profile["transform"] == first["transform"]
            and str(profile["crs"]) == str(first["crs"])
        )
        if not same:
            raise ValueError("Input rasters do not share the same grid.")


## Steps 01-05B - raster pipeline overview

The next cells define the core raster processing pipeline. Each step is defined as a function so the run-control cell can execute the pipeline in order and write a clear step-status table.

High-level flow:

1. Split multi-band LST stacks into one raster per year.
2. Build a multi-year LST baseline and calculate anomalies/z-scores.
3. Detect temporal and structural hotspots.
4. Create normalized LST, NDVI, and albedo inputs.
5. Build composite heat indices.
6. Prepare webapp-ready raster layers.
7. Build 1 km relationship tables for albedo, NDVI, and DeltaLST.

These function cells only define the logic. They do not process data until the `Run the pipeline` section is executed.


## Step 01 - Split Landsat stacks

Earth Engine exports LST as three multi-band GeoTIFF stacks: median LST, mean LST, and valid-observation count. Each band corresponds to one year.

This step reads those stacks and writes separate yearly single-band rasters into `raw/gee_lst`.

Inputs:

- `{CITY}_LST_{first_year}_{last_year}_summer_median_stack_30m.tif`
- `{CITY}_LST_{first_year}_{last_year}_summer_mean_stack_30m.tif`
- `{CITY}_LST_{first_year}_{last_year}_summer_validObs_stack_30m.tif`

Outputs:

- `{CITY}_LST_{year}_summer_median_30m.tif`
- `{CITY}_LST_{year}_summer_mean_30m.tif`
- `{CITY}_LST_{year}_summer_validObsCount_30m.tif`

The function checks that the number of bands equals the number of configured LST years. If this fails, the Earth Engine export year range and the notebook configuration do not match.


In [ ]:
def split_stack(stack_path: Path, years: tuple[int, ...], out_template: str, out_dir: Path, default_nodata: float):
    if not stack_path.exists():
        print(f"[WARN] File not found: {stack_path}")
        return

    with rasterio.open(stack_path) as src:
        band_count = src.count
        profile = src.profile.copy()
        src_nodata = src.nodata

        print("\n=== READING STACK ===")
        print(f"File: {stack_path.name}")
        print(f"Bands: {band_count}  Width: {src.width}  Height: {src.height}")
        print(f"CRS: {src.crs}  Nodata: {src_nodata}")

        if band_count != len(years):
            raise ValueError(
                f"Band count ({band_count}) != expected years ({len(years)}) "
                f"for {stack_path.name}"
            )

        out_profile = profile.copy()
        out_profile.update(count=1, dtype="float32", compress="deflate", nodata=default_nodata)

        for index, year in enumerate(years, start=1):
            arr = src.read(index).astype("float32")
            arr = mask_nodata_values(arr, src_nodata)
            out_arr = np.where(np.isfinite(arr), arr, default_nodata).astype("float32")
            out_path = out_dir / out_template.format(year=year)
            out_path.parent.mkdir(parents=True, exist_ok=True)
            with rasterio.open(out_path, "w", **out_profile) as dst:
                dst.write(out_arr, 1)
            print(f"[OK] band {index} -> {year} -> {out_path.name} | valid: {int(np.isfinite(arr).sum())}")


def step01_split_landsat_stack(cfg: PipelineConfig):
    print("=== STEP 01 - SPLIT LANDSAT MULTI-BAND STACKS ===")
    split_stack(cfg.raw_bands_dir / cfg.lst_stack_median_file, cfg.years_lst, cfg.lst_template, cfg.raw_lst_dir, cfg.default_nodata)
    split_stack(cfg.raw_bands_dir / cfg.lst_stack_mean_file, cfg.years_lst, cfg.lst_mean_template, cfg.raw_lst_dir, cfg.default_nodata)
    split_stack(cfg.raw_bands_dir / cfg.lst_stack_validobs_file, cfg.years_lst, cfg.lst_validobs_template, cfg.raw_lst_dir, cfg.default_nodata)
    print(f"\nYearly single-band files written to: {cfg.raw_lst_dir}")


## Step 02 - Temporal baseline

This step builds the multi-year summer LST baseline for each pixel.

For every pixel, it calculates:

- climatology mean: the average summer LST across all valid years;
- climatology standard deviation: how much that pixel varies year to year;
- valid year count: how many years had usable data;
- yearly anomaly: current year LST minus that pixel's climatology mean;
- yearly temporal z-score: anomaly divided by that pixel's climatology standard deviation.

Why this matters: a temporal z-score identifies places that are unusually hot compared with their own history, not just hot compared with the rest of the city.

Outputs are written to `intermediate/temporal`, and CSV summaries are written to `outputs/summaries`. Pixels with fewer than `min_valid_years` valid years are masked from climatology outputs.


In [ ]:
def step02_temporal_baseline_lst(cfg: PipelineConfig):
    print("=== STEP 02 - TEMPORAL BASELINE LST ===")
    arrays_profiles = []
    rows = []
    loaded_years = []
    for year in cfg.years_lst:
        path = cfg.raw_lst_dir / cfg.lst_template.format(year=year)
        if not path.exists():
            print(f"[WARN] Missing LST file for {year}: {path}")
            continue
        arr, profile = read_raster(path)
        arrays_profiles.append((arr, profile))
        loaded_years.append(year)
        st = raster_stats(arr)
        st.update({"year": year, "source_file": path.name})
        rows.append(st)

    if len(arrays_profiles) < 2:
        raise RuntimeError("Not enough yearly LST rasters found to build climatology.")

    ensure_same_grid(arrays_profiles)
    stack = np.stack([arr for arr, _ in arrays_profiles], axis=0)
    profile = arrays_profiles[0][1]

    valid_year_count = np.sum(np.isfinite(stack), axis=0).astype("float32")
    clim_mean = np.nanmean(stack, axis=0).astype("float32")
    clim_std = np.nanstd(stack, axis=0).astype("float32")
    clim_mean = np.where(valid_year_count >= cfg.min_valid_years, clim_mean, np.nan)
    clim_std = np.where(valid_year_count >= cfg.min_valid_years, clim_std, np.nan)

    write_raster(cfg.temporal_dir / "climatology_summer_mean_median_30m.tif", clim_mean, profile, cfg.default_nodata)
    write_raster(cfg.temporal_dir / "climatology_summer_std_median_30m.tif", clim_std, profile, cfg.default_nodata)
    write_raster(cfg.temporal_dir / "climatology_validYearCount_median_30m.tif", valid_year_count, profile, cfg.default_nodata)

    out_rows = []
    for (arr, _), year in zip(arrays_profiles, loaded_years):
        anomaly = (arr - clim_mean).astype("float32")
        z = np.where((np.isfinite(clim_std)) & (clim_std > 0), anomaly / clim_std, np.nan).astype("float32")
        write_raster(cfg.temporal_dir / f"anomaly_{year}_summer_median_30m.tif", anomaly, profile, cfg.default_nodata)
        write_raster(cfg.temporal_dir / f"zscore_{year}_summer_median_30m.tif", z, profile, cfg.default_nodata)
        out_rows.append({
            "year": year,
            "label": "summer",
            "source_file": cfg.lst_template.format(year=year),
            **{f"lst_{key}": value for key, value in raster_stats(arr).items()},
            **{f"anom_{key}": value for key, value in raster_stats(anomaly).items()},
            **{f"z_{key}": value for key, value in raster_stats(z).items()},
        })

    pd.DataFrame(out_rows).to_csv(cfg.summaries_dir / "LST_summary_anomalies_median_30m.csv", index=False)
    pd.DataFrame(rows).to_csv(cfg.summaries_dir / "LST_yearly_input_summary_median_30m.csv", index=False)
    print("[OK] Temporal baseline plus per-year anomaly/z-score written.")


## Step 03 - Temporal and structural hotspots

This step creates two different hotspot concepts for each year.

Temporal hotspots answer: where is this pixel unusually hot compared with its own multi-year history? A pixel is marked as temporal hotspot when its temporal z-score is above `hotspot_z_temp_threshold`.

Structural hotspots answer: where is this pixel among the hottest places in the city for that year? A pixel is marked as structural hotspot when its LST is above the configured citywide percentile, usually the 95th percentile.

The step also counts hotspot persistence across years. A persistence raster stores how many years each pixel was repeatedly classified as a hotspot.

Outputs are written to `intermediate/hotspots`, with summary CSVs in `outputs/summaries`.


In [ ]:
def step03_temporal_hotspots(cfg: PipelineConfig):
    print("=== STEP 03 - TEMPORAL AND STRUCTURAL HOTSPOTS ===")
    z_arrays, lst_arrays, years = [], [], []
    for year in cfg.years_lst:
        z_path = cfg.temporal_dir / f"zscore_{year}_summer_median_30m.tif"
        lst_path = cfg.raw_lst_dir / cfg.lst_template.format(year=year)
        if z_path.exists() and lst_path.exists():
            z_arrays.append(read_raster(z_path))
            lst_arrays.append(read_raster(lst_path))
            years.append(year)

    if not z_arrays:
        raise RuntimeError("No z-score rasters found.")

    ensure_same_grid(z_arrays)
    ensure_same_grid(lst_arrays)
    profile = z_arrays[0][1]

    period = f"{years[0]}_{years[-1]}"
    temporal_binary_stack, structural_binary_stack, summary_rows = [], [], []
    temporal_valid_any = np.zeros((profile["height"], profile["width"]), dtype=bool)
    structural_valid_any = np.zeros((profile["height"], profile["width"]), dtype=bool)

    for (z_arr, _), (lst_arr, _), year in zip(z_arrays, lst_arrays, years):
        valid_z = np.isfinite(z_arr)
        valid_lst = np.isfinite(lst_arr)

        hotspot_temp = np.full(z_arr.shape, np.nan, dtype="float32")
        hotspot_temp_strong = np.full(z_arr.shape, np.nan, dtype="float32")
        hotspot_temp[valid_z] = (z_arr[valid_z] > cfg.hotspot_z_temp_threshold).astype("float32")
        hotspot_temp_strong[valid_z] = (z_arr[valid_z] > cfg.hotspot_z_temp_strong_threshold).astype("float32")

        hotspot_struct = np.full(lst_arr.shape, np.nan, dtype="float32")
        if np.any(valid_lst):
            structural_threshold = np.nanpercentile(lst_arr[valid_lst], cfg.hotspot_percentile)
            hotspot_struct[valid_lst] = (lst_arr[valid_lst] > structural_threshold).astype("float32")
        else:
            structural_threshold = np.nan

        temporal_binary_stack.append(hotspot_temp)
        structural_binary_stack.append(hotspot_struct)
        temporal_valid_any |= valid_z
        structural_valid_any |= valid_lst

        write_raster(cfg.hotspots_dir / f"hotspot_temporal_{year}_zgt1p0.tif", hotspot_temp, profile, cfg.default_nodata)
        write_raster(cfg.hotspots_dir / f"hotspot_temporal_{year}_zgt1p5.tif", hotspot_temp_strong, profile, cfg.default_nodata)
        write_raster(cfg.hotspots_dir / f"hotspot_structural_{year}_top5pct.tif", hotspot_struct, profile, cfg.default_nodata)

        summary_rows.append({
            "year": year,
            "temporal_hotspot_fraction": float(np.nanmean(hotspot_temp[valid_z])) if np.any(valid_z) else np.nan,
            "temporal_hotspot_strong_fraction": float(np.nanmean(hotspot_temp_strong[valid_z])) if np.any(valid_z) else np.nan,
            "structural_hotspot_threshold": float(structural_threshold) if np.isfinite(structural_threshold) else np.nan,
            "structural_hotspot_fraction": float(np.nanmean(hotspot_struct[valid_lst])) if np.any(valid_lst) else np.nan,
        })

    persistence_temp = np.nansum(np.stack(temporal_binary_stack, axis=0), axis=0).astype("float32")
    persistence_struct = np.nansum(np.stack(structural_binary_stack, axis=0), axis=0).astype("float32")
    persistence_temp[~temporal_valid_any] = np.nan
    persistence_struct[~structural_valid_any] = np.nan

    write_raster(cfg.hotspots_dir / f"hotspot_temporal_persistence_{period}.tif", persistence_temp, profile, cfg.default_nodata)
    write_raster(cfg.hotspots_dir / f"hotspot_structural_persistence_{period}.tif", persistence_struct, profile, cfg.default_nodata)

    persistence_temp_ge = np.full_like(persistence_temp, np.nan, dtype="float32")
    persistence_temp_ge[temporal_valid_any] = (
        persistence_temp[temporal_valid_any] >= cfg.hotspot_persistence_threshold
    ).astype("float32")
    write_raster(
        cfg.hotspots_dir / f"hotspot_temporal_persistence_ge_{cfg.hotspot_persistence_threshold}.tif",
        persistence_temp_ge,
        profile,
        cfg.default_nodata,
    )

    pd.DataFrame(summary_rows).to_csv(cfg.summaries_dir / "temporal_hotspot_summary.csv", index=False)
    pd.DataFrame([
        {"metric": "temporal_persistence", **raster_stats(persistence_temp)},
        {"metric": "structural_persistence", **raster_stats(persistence_struct)},
    ]).to_csv(cfg.summaries_dir / "temporal_hotspot_persistence_summary.csv", index=False)
    print("[OK] Temporal hotspots and persistence written.")


## Step 04A - Spatial z-score

This step standardizes `webapp_year` LST against the citywide LST distribution for that same year.

Formula:

`spatial_z = (pixel_LST - city_mean_LST) / city_std_LST`

Why this matters: spatial z-score highlights pixels that are hot relative to other places in the same city during the same summer. This differs from the temporal z-score, which compares each pixel against its own history.

Output:

- `intermediate/spatial/zscore_spatial_{webapp_year}_summer_median_30m.tif`
- a summary CSV with city mean, city standard deviation, and raster statistics.


In [ ]:
def step04a_spatial_zscore(cfg: PipelineConfig):
    print("=== STEP 04A - SPATIAL Z-SCORE ===")
    lst_path = require_file(cfg.raw_lst_dir / cfg.lst_template.format(year=cfg.webapp_year), "WEBAPP_YEAR LST")
    arr, profile = read_raster(lst_path)
    vals = arr[np.isfinite(arr)]
    if vals.size == 0:
        raise ValueError(f"No valid LST pixels in {lst_path}")
    urban_mean = float(np.nanmean(vals))
    urban_std = float(np.nanstd(vals))
    z_spatial = np.full_like(arr, np.nan, dtype="float32")
    if urban_std > 0:
        z_spatial = ((arr - urban_mean) / urban_std).astype("float32")
        z_spatial[~np.isfinite(arr)] = np.nan

    write_raster(cfg.spatial_dir / f"zscore_spatial_{cfg.webapp_year}_summer_median_30m.tif", z_spatial, profile, cfg.default_nodata)
    pd.DataFrame([{
        "year": cfg.webapp_year,
        f"media_urbana_{cfg.webapp_year}": urban_mean,
        f"std_urbana_{cfg.webapp_year}": urban_std,
        **{f"zsp_{key}": value for key, value in raster_stats(z_spatial).items()},
    }]).to_csv(cfg.summaries_dir / f"zscore_spatial_{cfg.webapp_year}_summary.csv", index=False)
    print("[OK] Spatial z-score written.")


## Step 04B - Normalize inputs

This step prepares LST, NDVI, and albedo so they can be combined into composite indices.

Because these rasters can come from different sensors and grids, NDVI and albedo are first reprojected to match the LST grid. Then each input is min-max normalized to the range `[0, 1]`.

Why normalization is needed: LST is measured in degrees Celsius, NDVI usually ranges from about -1 to 1, and albedo has a different physical range. Composite formulas need comparable scales.

Outputs are written to `intermediate/normalized`:

- normalized LST;
- normalized NDVI;
- normalized albedo;
- a CSV summary of all three normalized rasters.


In [ ]:
def step04b_normalize(cfg: PipelineConfig):
    print("=== STEP 04B - NORMALIZE LST, NDVI, ALBEDO ===")
    lst, lst_profile = read_raster(require_file(cfg.raw_lst_dir / cfg.lst_template.format(year=cfg.webapp_year), "LST"))
    ndvi, ndvi_profile = read_raster(require_file(cfg.raw_ndvi_dir / cfg.ndvi_file, "NDVI"))
    albedo, albedo_profile = read_raster(require_file(cfg.raw_albedo_dir / cfg.albedo_file, "Albedo"))

    ndvi_match = reproject_to_match(ndvi, ndvi_profile, lst_profile, resampling=Resampling.bilinear)
    albedo_match = reproject_to_match(albedo, albedo_profile, lst_profile, resampling=Resampling.bilinear)

    lst_norm = minmax_normalize(lst)
    ndvi_norm = minmax_normalize(ndvi_match)
    albedo_norm = minmax_normalize(albedo_match)

    write_raster(cfg.normalized_dir / f"LST_{cfg.webapp_year}_norm_30m.tif", lst_norm, lst_profile, cfg.default_nodata)
    write_raster(cfg.normalized_dir / f"NDVI_{cfg.webapp_year}_norm_30m.tif", ndvi_norm, lst_profile, cfg.default_nodata)
    write_raster(cfg.normalized_dir / f"Albedo_{cfg.webapp_year}_norm_30m.tif", albedo_norm, lst_profile, cfg.default_nodata)

    pd.DataFrame([
        {"name": f"LST_norm_{cfg.webapp_year}_30m", **raster_stats(lst_norm)},
        {"name": f"NDVI_norm_{cfg.webapp_year}_30m", **raster_stats(ndvi_norm)},
        {"name": f"Albedo_norm_{cfg.webapp_year}_30m", **raster_stats(albedo_norm)},
    ]).to_csv(cfg.summaries_dir / f"normalized_{cfg.webapp_year}_summary.csv", index=False)
    print("[OK] Normalized rasters written.")


## Step 04C - Composite indices

This step combines normalized LST, NDVI, and albedo into three webapp-friendly heat indicators.

Indices:

- `HVI = LST_norm - NDVI_norm`: hotter and less vegetated areas score higher.
- `HRI = LST_norm - Albedo_norm`: hotter and darker/less reflective areas score higher.
- `UHEI = LST_norm + (1 - NDVI_norm) + (1 - Albedo_norm)`: a combined urban heat exposure indicator.

These indices are descriptive indicators, not physical temperature measurements. They are useful for ranking and comparing areas within the same city/year.

Outputs are written to `intermediate/composites`, with summary statistics in `outputs/summaries`.


In [ ]:
def step04c_composite_indices(cfg: PipelineConfig):
    print("=== STEP 04C - COMPOSITE INDICES ===")
    lst, profile = read_raster(require_file(cfg.normalized_dir / f"LST_{cfg.webapp_year}_norm_30m.tif", "normalized LST"))
    ndvi, _ = read_raster(require_file(cfg.normalized_dir / f"NDVI_{cfg.webapp_year}_norm_30m.tif", "normalized NDVI"))
    albedo, _ = read_raster(require_file(cfg.normalized_dir / f"Albedo_{cfg.webapp_year}_norm_30m.tif", "normalized albedo"))

    common = np.isfinite(lst) & np.isfinite(ndvi) & np.isfinite(albedo)

    hvi = np.full_like(lst, np.nan, dtype="float32")
    hri = np.full_like(lst, np.nan, dtype="float32")
    uhei = np.full_like(lst, np.nan, dtype="float32")

    hvi[common] = lst[common] - ndvi[common]
    hri[common] = lst[common] - albedo[common]
    uhei[common] = lst[common] + (1.0 - ndvi[common]) + (1.0 - albedo[common])

    write_raster(cfg.composites_dir / f"HVI_{cfg.webapp_year}_summer_30m.tif", hvi, profile, cfg.default_nodata)
    write_raster(cfg.composites_dir / f"HRI_{cfg.webapp_year}_summer_30m.tif", hri, profile, cfg.default_nodata)
    write_raster(cfg.composites_dir / f"UHEI_{cfg.webapp_year}_summer_30m.tif", uhei, profile, cfg.default_nodata)

    pd.DataFrame([
        {"name": f"HVI_{cfg.webapp_year}_summer_30m", **raster_stats(hvi)},
        {"name": f"HRI_{cfg.webapp_year}_summer_30m", **raster_stats(hri)},
        {"name": f"UHEI_{cfg.webapp_year}_summer_30m", **raster_stats(uhei)},
    ]).to_csv(cfg.summaries_dir / f"composite_indices_{cfg.webapp_year}_summary.csv", index=False)
    print("[OK] HVI/HRI/UHEI rasters written.")


## Step 04D - Hotspot type classes

This step combines the structural and temporal hotspot definitions for `webapp_year` into one categorical raster.

Classes:

- `0`: neither structural nor anomalous;
- `1`: structural only, meaning consistently one of the hottest places in the city;
- `2`: anomalous only, meaning unusually hot compared with its own history;
- `3`: both structural and anomalous.

This is useful for interpretation. A structural-only hotspot may represent a persistently hot urban surface, while an anomalous-only hotspot may represent a recent or unusual heat condition.

Output is written to `intermediate/hotspots`, and a summary CSV reports pixel counts for each class.


In [ ]:
def step04d_hotspot_types(cfg: PipelineConfig):
    print("=== STEP 04D - STRUCTURAL VS ANOMALOUS HOTSPOT TYPES ===")
    lst, lst_profile = read_raster(require_file(cfg.raw_lst_dir / cfg.lst_template.format(year=cfg.webapp_year), "LST"))
    z_temp, z_temp_profile = read_raster(require_file(cfg.temporal_dir / f"zscore_{cfg.webapp_year}_summer_median_30m.tif", "temporal z-score"))
    ensure_same_grid([(lst, lst_profile), (z_temp, z_temp_profile)])

    valid_lst = np.isfinite(lst)
    if not np.any(valid_lst):
        raise ValueError("No valid LST pixels available for hotspot classification.")
    structural_threshold = np.nanpercentile(lst[valid_lst], cfg.hotspot_percentile)
    common = np.isfinite(lst) & np.isfinite(z_temp)
    structural = common & (lst > structural_threshold)
    anomalous = common & (z_temp > cfg.hotspot_z_temp_threshold)

    out = np.full(lst.shape, np.nan, dtype="float32")
    out[common] = 0
    out[structural & ~anomalous] = 1
    out[~structural & anomalous] = 2
    out[structural & anomalous] = 3

    write_raster(cfg.hotspots_dir / f"hotspot_structural_vs_anomalous_new_{cfg.webapp_year}.tif", out, lst_profile, cfg.default_nodata)
    pd.DataFrame([{
        "year": cfg.webapp_year,
        "structural_percentile": cfg.hotspot_percentile,
        "structural_threshold": float(structural_threshold),
        "temporal_z_threshold": cfg.hotspot_z_temp_threshold,
        "valid_pixel_count": int(np.sum(common)),
        "structural_count": int(np.sum(structural)),
        "anomalous_count": int(np.sum(anomalous)),
        "none_count": int(np.sum(out == 0)),
        "structural_only_count": int(np.sum(out == 1)),
        "anomalous_only_count": int(np.sum(out == 2)),
        "both_count": int(np.sum(out == 3)),
    }]).to_csv(cfg.summaries_dir / f"hotspot_structural_vs_anomalous_new_{cfg.webapp_year}_summary.csv", index=False)
    print(f"[OK] Structural-vs-anomalous classification for {cfg.webapp_year} written.")


## Step 04E - All-year hotspot persistence

This step summarizes hotspot behavior across the full LST period.

For each pixel, it counts:

- how many years it was a structural hotspot;
- how many years it was an anomalous temporal hotspot.

Both counts are packed into one raster using this formula:

`packed = structural_count + 100 * anomalous_count`

Downstream code can recover the two counts from the packed value. This compact representation keeps two related persistence channels in a single-band raster.

Outputs include the packed persistence raster and yearly/persistence summary CSVs.


In [ ]:
PACK_BASE = 100


def load_yearly_hotspot_masks(cfg: PipelineConfig):
    structural_arrays, anomalous_arrays, years = [], [], []
    for year in cfg.years_lst:
        structural_path = cfg.hotspots_dir / f"hotspot_structural_{year}_top5pct.tif"
        anomalous_path = cfg.hotspots_dir / f"hotspot_temporal_{year}_zgt1p0.tif"
        if structural_path.exists() and anomalous_path.exists():
            structural_arrays.append(read_raster(structural_path))
            anomalous_arrays.append(read_raster(anomalous_path))
            years.append(year)

    if not years:
        raise RuntimeError("No yearly structural/anomalous hotspot rasters found.")

    ensure_same_grid(structural_arrays + anomalous_arrays)
    return structural_arrays, anomalous_arrays, years, structural_arrays[0][1]


def pack_counts(structural_count, anomalous_count):
    return (structural_count + (PACK_BASE * anomalous_count)).astype("float32")


def step04e_composite_allyears(cfg: PipelineConfig):
    print("=== STEP 04E - ALL-YEARS STRUCTURAL/ANOMALOUS PERSISTENCE ===")
    structural_arrays, anomalous_arrays, years, profile = load_yearly_hotspot_masks(cfg)
    period_label = f"{min(years)}_{max(years)}"

    structural_stack, anomalous_stack, both_stack = [], [], []
    valid_any = np.zeros((profile["height"], profile["width"]), dtype=bool)
    yearly_rows = []

    for year, (structural_arr, _), (anomalous_arr, _) in zip(years, structural_arrays, anomalous_arrays):
        common = np.isfinite(structural_arr) & np.isfinite(anomalous_arr)
        structural = common & (structural_arr > 0)
        anomalous = common & (anomalous_arr > 0)

        both = np.where(structural & anomalous, 1, 0).astype("float32")
        structural_flag = np.where(structural, 1, 0).astype("float32")
        anomalous_flag = np.where(anomalous, 1, 0).astype("float32")

        structural_stack.append(structural_flag)
        anomalous_stack.append(anomalous_flag)
        both_stack.append(both)
        valid_any |= common

        yearly_rows.append({
            "year": year,
            "valid_pixel_count": int(np.sum(common)),
            "none_count": int(np.sum(common & ~structural & ~anomalous)),
            "structural_count": int(np.sum(structural_flag)),
            "anomalous_count": int(np.sum(anomalous_flag)),
            "both_count": int(np.sum(both)),
        })

    structural_count = np.sum(np.stack(structural_stack, axis=0), axis=0).astype("float32")
    anomalous_count = np.sum(np.stack(anomalous_stack, axis=0), axis=0).astype("float32")
    both_count = np.sum(np.stack(both_stack, axis=0), axis=0).astype("float32")
    packed = pack_counts(structural_count, anomalous_count)
    packed[~valid_any] = np.nan

    write_raster(cfg.hotspots_dir / f"hotspot_structural_vs_anomalous_persistence_new_{period_label}.tif", packed, profile, cfg.default_nodata)

    pd.DataFrame(yearly_rows).to_csv(cfg.summaries_dir / f"hotspot_structural_vs_anomalous_{period_label}_yearly_counts_new.csv", index=False)
    pd.DataFrame([{
        "period": period_label,
        "year_count": len(years),
        "encoding_base": PACK_BASE,
        "valid_pixel_count": int(np.sum(valid_any)),
        "structural_total_flags": int(np.nansum(structural_count)),
        "anomalous_total_flags": int(np.nansum(anomalous_count)),
        "both_total_flags": int(np.nansum(both_count)),
        "structural_pixels_ever": int(np.sum(structural_count > 0)),
        "anomalous_pixels_ever": int(np.sum(anomalous_count > 0)),
        "both_pixels_ever": int(np.sum(both_count > 0)),
        "packed_value_formula": f"structural + {PACK_BASE}*anomalous",
        "both_rule": "years with both structural and anomalous increment both channels",
    }]).to_csv(cfg.summaries_dir / f"hotspot_structural_vs_anomalous_persistence_{period_label}_summary.csv", index=False)
    print("[OK] All-years packed structural/anomalous persistence written.")


## Step 04F - Albedo and DeltaLST relationship

This step compares 30 m Landsat albedo with 1 km MODIS day-night DeltaLST.

Because albedo and DeltaLST are on different grids, albedo is average-resampled to the MODIS 1 km grid. The step then builds a table where each row is one overlapping 1 km cell.

Outputs:

- an upscaled 1 km albedo raster;
- a CSV pair table with albedo and DeltaLST per grid cell;
- a CSV with relationship statistics such as correlation and slope;
- a scatter plot showing the relationship.

This step is exploratory: it helps assess whether more reflective surfaces are associated with lower or higher day-night LST differences in the study area.


In [ ]:
def upscale_to_match_mean(src_arr, src_profile, dst_profile, default_nodata=-9999.0):
    src_nodata = src_profile.get("nodata")
    src_fill = src_nodata if src_nodata is not None else default_nodata
    src_data = np.where(np.isfinite(src_arr), src_arr, src_fill).astype("float32")
    dst = np.full((dst_profile["height"], dst_profile["width"]), default_nodata, dtype="float32")
    reproject(
        source=src_data,
        destination=dst,
        src_transform=src_profile["transform"],
        src_crs=src_profile["crs"],
        src_nodata=src_fill,
        dst_transform=dst_profile["transform"],
        dst_crs=dst_profile["crs"],
        dst_nodata=default_nodata,
        resampling=Resampling.average,
    )
    dst[dst == default_nodata] = np.nan
    return dst


def build_pair_table(albedo_1km, deltalst_1km, transform):
    valid = np.isfinite(albedo_1km) & np.isfinite(deltalst_1km)
    rows, cols = np.where(valid)
    if rows.size == 0:
        raise ValueError("No overlapping valid cells were found between albedo and DeltaLST.")
    xs, ys = rasterio.transform.xy(transform, rows, cols, offset="center")
    return pd.DataFrame({
        "row": rows.astype(int),
        "col": cols.astype(int),
        "x": np.asarray(xs, dtype="float64"),
        "y": np.asarray(ys, dtype="float64"),
        "albedo_mean_1km": albedo_1km[valid].astype("float64"),
        "delta_lst_1km": deltalst_1km[valid].astype("float64"),
    })


def compute_relationship_stats(data: pd.DataFrame):
    x = data["albedo_mean_1km"].to_numpy(dtype="float64")
    y = data["delta_lst_1km"].to_numpy(dtype="float64")
    if x.size < 2 or np.allclose(x, x[0]):
        slope = intercept = pearson_r = np.nan
    else:
        slope, intercept = np.polyfit(x, y, 1)
        pearson_r = float(np.corrcoef(x, y)[0, 1])
    return {
        "count": int(x.size),
        "albedo_mean": float(np.nanmean(x)),
        "albedo_std": float(np.nanstd(x)),
        "delta_lst_mean": float(np.nanmean(y)),
        "delta_lst_std": float(np.nanstd(y)),
        "pearson_r": pearson_r,
        "slope": float(slope) if np.isfinite(slope) else np.nan,
        "intercept": float(intercept) if np.isfinite(intercept) else np.nan,
    }


def make_albedo_deltalst_plot(data: pd.DataFrame, stats: dict, out_path: Path, year: int):
    x = data["albedo_mean_1km"].to_numpy(dtype="float64")
    y = data["delta_lst_1km"].to_numpy(dtype="float64")
    fig, ax = plt.subplots(figsize=(8, 6), dpi=180)
    ax.scatter(x, y, s=28, alpha=0.75, color="#1f6f8b", edgecolors="white", linewidths=0.35)
    if np.isfinite(stats["slope"]) and np.isfinite(stats["intercept"]):
        x_line = np.linspace(np.nanmin(x), np.nanmax(x), 200)
        ax.plot(x_line, stats["slope"] * x_line + stats["intercept"], color="#c84b31", linewidth=2.0)
    ax.set_title(f"Albedo vs DeltaLST, summer {year}")
    ax.set_xlabel("Albedo (mean aggregated to 1 km)")
    ax.set_ylabel("DeltaLST (1 km)")
    ax.grid(True, linestyle="--", linewidth=0.6, alpha=0.35)
    summary = "\n".join([
        f"n = {stats['count']}",
        f"r = {stats['pearson_r']:.3f}" if np.isfinite(stats["pearson_r"]) else "r = n/a",
        f"slope = {stats['slope']:.3f}" if np.isfinite(stats["slope"]) else "slope = n/a",
    ])
    ax.text(
        0.02, 0.98, summary, transform=ax.transAxes, va="top", ha="left", fontsize=10,
        bbox={"boxstyle": "round,pad=0.35", "facecolor": "white", "alpha": 0.9, "edgecolor": "#bbbbbb"},
    )
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(out_path, bbox_inches="tight")
    plt.close(fig)


def step04f_albedo_deltalst_graph(cfg: PipelineConfig):
    print("=== STEP 04F - ALBEDO VS DELTALST RELATIONSHIP ===")
    albedo_30m, albedo_profile = read_raster(require_file(cfg.raw_albedo_dir / cfg.albedo_file, "albedo"))
    deltalst_1km, deltalst_profile = read_raster(require_file(cfg.raw_modis_dir / cfg.deltalst_file, "DeltaLST"))

    albedo_1km = upscale_to_match_mean(albedo_30m, albedo_profile, deltalst_profile, cfg.default_nodata)
    upscaled_albedo_path = cfg.spatial_dir / f"Albedo_{cfg.webapp_year}_summer_mean_1km_from30m.tif"
    write_raster(upscaled_albedo_path, albedo_1km, deltalst_profile, cfg.default_nodata)

    pair_table = build_pair_table(albedo_1km, deltalst_1km, deltalst_profile["transform"])
    pair_table_path = cfg.summaries_dir / f"albedo_deltalst_{cfg.webapp_year}_1km_pairs.csv"
    pair_table.to_csv(pair_table_path, index=False)

    stats = compute_relationship_stats(pair_table)
    pd.DataFrame([stats]).to_csv(cfg.summaries_dir / f"albedo_deltalst_{cfg.webapp_year}_1km_stats.csv", index=False)

    make_albedo_deltalst_plot(pair_table, stats, cfg.summaries_dir / f"albedo_vs_deltalst_{cfg.webapp_year}_1km.png", cfg.webapp_year)
    print(f"[OK] Albedo-vs-DeltaLST pair table, stats and plot written for {cfg.webapp_year}.")


## Step 05 - Prepare webapp rasters

This step collects the curated raster layers that the webapp needs and writes them into `outputs/webapp_rasters`.

It standardizes nodata handling and writes a manifest file listing the available raster layers. NDVI is also reprojected to the LST grid so the webapp gets a 30 m NDVI layer aligned with the other 30 m products.

Typical webapp layers include:

- LST;
- anomaly;
- temporal and spatial z-scores;
- NDVI, albedo, and DeltaLST;
- HVI, HRI, and UHEI;
- hotspot masks and persistence layers;
- climatology mean.

The manifest is written to `outputs/metadata/webapp_raster_manifest.json`.


In [ ]:
def webapp_layer_map(cfg: PipelineConfig) -> dict[str, Path]:
    period = cfg.period_label
    return {
        f"LST_{cfg.webapp_year}_summer_30m.tif": cfg.raw_lst_dir / cfg.lst_template.format(year=cfg.webapp_year),
        f"anomaly_{cfg.webapp_year}_summer_30m.tif": cfg.temporal_dir / f"anomaly_{cfg.webapp_year}_summer_median_30m.tif",
        f"zscore_temporal_{cfg.webapp_year}_summer_30m.tif": cfg.temporal_dir / f"zscore_{cfg.webapp_year}_summer_median_30m.tif",
        f"zscore_spatial_{cfg.webapp_year}_summer_30m.tif": cfg.spatial_dir / f"zscore_spatial_{cfg.webapp_year}_summer_median_30m.tif",
        f"NDVI_{cfg.webapp_year}_summer_30m.tif": cfg.raw_ndvi_dir / cfg.ndvi_file,
        f"Albedo_{cfg.webapp_year}_summer_30m.tif": cfg.raw_albedo_dir / cfg.albedo_file,
        f"DeltaLST_{cfg.webapp_year}_summer_1km.tif": cfg.raw_modis_dir / cfg.deltalst_file,
        f"HVI_{cfg.webapp_year}_summer_30m.tif": cfg.composites_dir / f"HVI_{cfg.webapp_year}_summer_30m.tif",
        f"HRI_{cfg.webapp_year}_summer_30m.tif": cfg.composites_dir / f"HRI_{cfg.webapp_year}_summer_30m.tif",
        f"UHEI_{cfg.webapp_year}_summer_30m.tif": cfg.composites_dir / f"UHEI_{cfg.webapp_year}_summer_30m.tif",
        f"hotspot_temporal_{cfg.webapp_year}_summer_30m.tif": cfg.hotspots_dir / f"hotspot_temporal_{cfg.webapp_year}_zgt1p0.tif",
        f"hotspot_temporal_persistence_{period}.tif": cfg.hotspots_dir / f"hotspot_temporal_persistence_{period}.tif",
        f"hotspot_structural_persistence_{period}.tif": cfg.hotspots_dir / f"hotspot_structural_persistence_{period}.tif",
        f"hotspot_structural_vs_anomalous_{cfg.webapp_year}.tif": cfg.hotspots_dir / f"hotspot_structural_vs_anomalous_new_{cfg.webapp_year}.tif",
        f"climatology_mean_{period}_30m.tif": cfg.temporal_dir / "climatology_summer_mean_median_30m.tif",
    }


def prepare_raster(src: Path, dst: Path, reference_profile=None, default_nodata=-9999.0):
    arr, profile = read_raster(src)
    if reference_profile is not None:
        arr = reproject_to_match(arr, profile, reference_profile, resampling=Resampling.bilinear)
        profile = reference_profile
    write_raster(dst, arr, profile, default_nodata)


def step05_prepare_webapp_rasters(cfg: PipelineConfig):
    print("=== STEP 05 - PREPARE WEBAPP RASTERS ===")
    lst_reference_path = cfg.raw_lst_dir / cfg.lst_template.format(year=cfg.webapp_year)
    lst_reference_profile = None
    if lst_reference_path.exists():
        with rasterio.open(lst_reference_path) as src:
            lst_reference_profile = src.profile.copy()

    reproject_to_lst_grid = {f"NDVI_{cfg.webapp_year}_summer_30m.tif"}
    manifest = []
    for out_name, src in webapp_layer_map(cfg).items():
        if not src.exists():
            print(f"[WARN] Missing source raster: {src}")
            continue
        if out_name in reproject_to_lst_grid and lst_reference_profile is None:
            print(f"[WARN] Cannot reproject {src.name}: missing LST reference raster.")
            continue
        dst = cfg.webapp_rasters_dir / out_name
        prepare_raster(
            src,
            dst,
            reference_profile=lst_reference_profile if out_name in reproject_to_lst_grid else None,
            default_nodata=cfg.default_nodata,
        )
        manifest.append({"layer_id": out_name.replace(".tif", ""), "filename": out_name, "type": "raster"})

    (cfg.metadata_dir / "webapp_raster_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    print("[OK] Webapp raster assets prepared.")


## Step 05B - Add NDVI to 1 km pair table

This step extends the 1 km albedo/DeltaLST table from Step 04F by adding mean NDVI for each same 1 km MODIS grid cell.

Why this is useful: it creates one consistent table for comparing surface reflectance, vegetation, and thermal behavior at the same spatial resolution.

Output:

- `outputs/summaries/albedo_ndvi_delta_{webapp_year}_1km_pairs.csv`

The output drops rows where NDVI is missing after reprojection. The printed message reports how many rows were kept.


In [ ]:
def step05b_albedo_ndvi_delta_pairs(cfg: PipelineConfig):
    print("=== STEP 05B - ADD NDVI TO ALBEDO/DELTALST PAIRS ===")
    pairs_in_path = cfg.summaries_dir / f"albedo_deltalst_{cfg.webapp_year}_1km_pairs.csv"
    pairs_out_path = cfg.summaries_dir / f"albedo_ndvi_delta_{cfg.webapp_year}_1km_pairs.csv"
    if not pairs_in_path.exists():
        raise FileNotFoundError(f"Run step04f first - missing {pairs_in_path}")

    pairs = pd.read_csv(pairs_in_path)
    ndvi, ndvi_profile = read_raster(require_file(cfg.raw_ndvi_dir / cfg.ndvi_file, "NDVI"))
    _, deltalst_profile = read_raster(require_file(cfg.raw_modis_dir / cfg.deltalst_file, "DeltaLST"))
    ndvi_1km = reproject_to_match(ndvi, ndvi_profile, deltalst_profile, resampling=Resampling.average)

    rows = pairs["row"].to_numpy(dtype=int)
    cols = pairs["col"].to_numpy(dtype=int)
    in_bounds = (rows >= 0) & (rows < ndvi_1km.shape[0]) & (cols >= 0) & (cols < ndvi_1km.shape[1])
    ndvi_means = np.full(len(pairs), np.nan, dtype="float64")
    ndvi_means[in_bounds] = ndvi_1km[rows[in_bounds], cols[in_bounds]]
    pairs["ndvi_mean_1km"] = ndvi_means

    out = pairs[["row", "col", "x", "y", "albedo_mean_1km", "ndvi_mean_1km", "delta_lst_1km"]].dropna(subset=["ndvi_mean_1km"])
    out.to_csv(pairs_out_path, index=False)

    if len(out):
        dropped = len(pairs) - len(out)
        print(
            f"[OK] Wrote {len(out)} rows to {pairs_out_path} ({dropped} cells dropped due to NaN NDVI). "
            f"NDVI mean={out['ndvi_mean_1km'].mean():.3f} "
            f"median={out['ndvi_mean_1km'].median():.3f} "
            f"range=[{out['ndvi_mean_1km'].min():.3f}, {out['ndvi_mean_1km'].max():.3f}]"
        )
    else:
        print("[WARN] No rows with valid NDVI - output is empty.")


## Steps 06-07 - vectors, polygon stats, optional clipping

The previous steps create raster outputs. Steps 06 and 07 work with vector boundaries and district/statistical-area polygons.

Step 06A prepares a city boundary file. If download URLs are configured, it can download vectors. If no URLs are configured, it expects the statistical-area GeoJSON to already exist locally and dissolves it into one boundary polygon.

Step 06B summarizes raster values inside each district or statistical area. This creates enriched vector layers for the webapp.

Step 07 optionally clips all webapp rasters to the city boundary. Some webapps prefer rectangular rasters plus a boundary overlay; others prefer physically clipped rasters. The run-control flags decide whether this optional clipping runs.


## Vector helper dependencies

This cell defines small guard functions for vector-related dependencies.

If `geopandas`, `rasterstats`, or `requests` are missing, the notebook raises a clear error only when a vector step actually needs that package. This keeps the raster pipeline understandable even in environments where vector dependencies are not installed yet.


In [ ]:
def require_geopandas():
    if gpd is None:
        raise ImportError("geopandas is required for vector steps. Install the project requirements first.")


def require_requests():
    if requests is None:
        raise ImportError("requests is required for vector downloads. Install the project requirements first.")


def require_rasterstats():
    if zonal_stats is None:
        raise ImportError("rasterstats is required for polygon statistics. Install the project requirements first.")


## Step 06A - Download vectors or build boundary

This step prepares `boundary_file`, which is required for clipping and useful for map display.

Two modes are supported:

1. Download mode: if `CFG.vector_download_urls` contains URLs, files are downloaded into `vector/`.
2. Manual-file mode: if no URLs are configured, the notebook expects `boundary_source_filename` to already exist in `vector/`.

After the source vector exists, the step dissolves all polygons into a single city boundary and writes `{city_slug}_boundary.geojson`.


In [ ]:
def download_file(url: str, out_path: Path) -> None:
    require_requests()
    print(f"Downloading: {url}")
    response = requests.get(url, timeout=120)
    response.raise_for_status()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_bytes(response.content)
    print(f"[OK] Saved: {out_path}")


def build_boundary_from_polygons(src_path: Path, out_path: Path, city: str) -> None:
    require_geopandas()
    gdf = gpd.read_file(src_path)
    dissolved = gdf.dissolve().reset_index(drop=True)
    dissolved["name"] = city
    out_path.parent.mkdir(parents=True, exist_ok=True)
    dissolved.to_file(out_path, driver="GeoJSON")
    print(f"[OK] Boundary written: {out_path}")


def step06a_download_vectors(cfg: PipelineConfig):
    print("=== STEP 06A - DOWNLOAD OR BUILD VECTOR BOUNDARY ===")
    if cfg.vector_download_urls:
        for filename, url in cfg.vector_download_urls.items():
            download_file(url, cfg.vector_dir / filename)
    else:
        print("[INFO] vector_download_urls is empty - skipping downloads.")
        print(f"       Make sure {cfg.boundary_source_filename} is already in {cfg.vector_dir}.")

    src = cfg.vector_dir / cfg.boundary_source_filename
    if not src.exists():
        raise FileNotFoundError(
            f"Cannot build boundary: {src} not found. "
            "Set vector_download_urls or place the source file manually."
        )
    build_boundary_from_polygons(src, cfg.boundary_file, cfg.city)

    print("\nFiles in vector directory:")
    for path in sorted(cfg.vector_dir.glob("*")):
        print(" -", path.name)


## Step 06B - Polygon statistics

This step computes zonal statistics for each district/statistical-area polygon.

For each polygon, the notebook calculates mean, median, and max values for selected webapp rasters, including LST, anomaly, z-scores, and composite heat indices.

Inputs:

- `districts_file`, usually `{city_slug}_statistical_areas.geojson`;
- prepared webapp rasters from Step 05.

Outputs:

- `outputs/webapp_vectors/districts_enriched_{webapp_year}.gpkg`
- `outputs/webapp_vectors/districts_enriched_{webapp_year}.geojson`

These enriched vector files are useful for district-level maps, tables, and dashboards.


In [ ]:
def polygon_stats_targets(cfg: PipelineConfig) -> dict[str, Path]:
    y = cfg.webapp_year
    return {
        f"lst_mean_{y}": cfg.webapp_rasters_dir / f"LST_{y}_summer_30m.tif",
        f"anomaly_mean_{y}": cfg.webapp_rasters_dir / f"anomaly_{y}_summer_30m.tif",
        f"ztemp_mean_{y}": cfg.webapp_rasters_dir / f"zscore_temporal_{y}_summer_30m.tif",
        f"zsp_mean_{y}": cfg.webapp_rasters_dir / f"zscore_spatial_{y}_summer_30m.tif",
        f"uhei_mean_{y}": cfg.webapp_rasters_dir / f"UHEI_{y}_summer_30m.tif",
        f"hvi_mean_{y}": cfg.webapp_rasters_dir / f"HVI_{y}_summer_30m.tif",
        f"hri_mean_{y}": cfg.webapp_rasters_dir / f"HRI_{y}_summer_30m.tif",
    }


def step06b_polygon_stats(cfg: PipelineConfig):
    print("=== STEP 06B - POLYGON STATS ===")
    require_geopandas()
    require_rasterstats()
    districts_path = require_file(cfg.districts_file, "district/statistical areas")
    gdf = gpd.read_file(districts_path)

    for col, raster_path in polygon_stats_targets(cfg).items():
        if not raster_path.exists():
            print(f"[WARN] Missing raster: {raster_path}")
            continue
        stats = zonal_stats(gdf, raster_path, stats=["mean", "median", "max"], geojson_out=False, nodata=cfg.default_nodata)
        df = pd.DataFrame(stats)
        gdf[col] = df["mean"]
        gdf[col.replace("_mean_", "_median_")] = df["median"]
        gdf[col.replace("_mean_", "_max_")] = df["max"]

    out_gpkg = cfg.webapp_vectors_dir / f"districts_enriched_{cfg.webapp_year}.gpkg"
    out_geojson = cfg.webapp_vectors_dir / f"districts_enriched_{cfg.webapp_year}.geojson"
    cfg.webapp_vectors_dir.mkdir(parents=True, exist_ok=True)
    gdf.to_file(out_gpkg, driver="GPKG")
    gdf.to_file(out_geojson, driver="GeoJSON")
    print("[OK] Polygon stats written:", out_gpkg)


## Step 07 - Optional clipping to boundary

This step clips every GeoTIFF in a folder to the city boundary polygon.

By default, it clips `outputs/webapp_rasters` and writes a sibling folder named like `webapp_rasters_{city_slug}_clipped`.

Run this step when you need rasters that physically cover only the municipality. If your webapp handles rectangular rasters and displays a boundary overlay, clipped rasters may be optional.

The function preserves raster metadata where possible and writes a valid raster mask for the clipped area.


In [ ]:
def validate_input_dir(path: Path) -> Path:
    resolved = Path(path).expanduser().resolve()
    if not resolved.exists() or not resolved.is_dir():
        raise FileNotFoundError(f"Input directory not found: {resolved}")
    return resolved


def resolve_output_dir(input_dir: Path, output_dir: Path | None, city_slug: str) -> Path:
    if output_dir is not None:
        return Path(output_dir).expanduser().resolve()
    return input_dir.parent / f"{input_dir.name}_{city_slug}_clipped"


def load_boundary(boundary_path: Path):
    require_geopandas()
    resolved = Path(boundary_path).expanduser().resolve()
    if not resolved.exists():
        raise FileNotFoundError(f"Boundary file not found: {resolved}")
    boundary = gpd.read_file(resolved)
    boundary = boundary.loc[boundary.geometry.notna() & ~boundary.geometry.is_empty].copy()
    if boundary.empty:
        raise ValueError(f"Boundary file contains no valid geometries: {resolved}")
    if boundary.crs is None:
        raise ValueError(f"Boundary file has no CRS: {resolved}")
    return boundary


def iter_raster_paths(input_dir: Path, output_dir: Path) -> list[Path]:
    rasters = []
    output_inside_input = False
    try:
        output_dir.relative_to(input_dir)
        output_inside_input = True
    except ValueError:
        pass

    for path in input_dir.rglob("*"):
        if not path.is_file() or path.suffix.lower() not in {".tif", ".tiff"}:
            continue
        if output_inside_input:
            try:
                path.relative_to(output_dir)
                continue
            except ValueError:
                pass
        rasters.append(path)
    return sorted(rasters)


def reproject_boundary(boundary, raster_crs):
    boundary_in_raster_crs = boundary.to_crs(raster_crs)
    return [geom.__geo_interface__ for geom in boundary_in_raster_crs.geometry if geom is not None]


def clip_raster_to_boundary(src_path: Path, dst_path: Path, boundary) -> None:
    with rasterio.open(src_path) as src:
        if src.crs is None:
            raise ValueError(f"Raster has no CRS: {src_path}")
        shapes = reproject_boundary(boundary, src.crs)
        clipped, transform = rasterio_mask(src, shapes, crop=True, filled=False)
        profile = src.profile.copy()
        profile.update(
            height=clipped.shape[1],
            width=clipped.shape[2],
            transform=transform,
            compress=profile.get("compress", "deflate"),
        )
        fill_value = src.nodata if src.nodata is not None else 0
        if src.nodata is None:
            profile.pop("nodata", None)
        else:
            profile["nodata"] = src.nodata
        mask_array = np.ma.getmaskarray(clipped)
        valid_mask = (~np.any(mask_array, axis=0)).astype("uint8") * 255
        data = clipped.filled(fill_value)
        dst_path.parent.mkdir(parents=True, exist_ok=True)
        with rasterio.open(dst_path, "w", **profile) as dst:
            dst.write(data)
            dst.write_mask(valid_mask)


def step07_clip_to_boundary(
    cfg: PipelineConfig,
    input_dir: Path | None = None,
    output_dir: Path | None = None,
    boundary_path: Path | None = None,
    overwrite: bool = False,
):
    print("=== STEP 07 - CLIP RASTERS TO BOUNDARY ===")
    input_dir = validate_input_dir(input_dir or cfg.webapp_rasters_dir)
    output_dir = resolve_output_dir(input_dir, output_dir, cfg.city_slug)
    boundary = load_boundary(boundary_path or cfg.boundary_file)
    raster_paths = iter_raster_paths(input_dir, output_dir)

    if not raster_paths:
        print(f"[WARN] No GeoTIFF files found in {input_dir}")
        return

    written = skipped = failed = 0
    for src_path in raster_paths:
        dst_path = output_dir / src_path.relative_to(input_dir)
        if dst_path.exists() and not overwrite:
            skipped += 1
            print(f"[SKIP] Already exists: {dst_path}")
            continue
        try:
            clip_raster_to_boundary(src_path, dst_path, boundary)
        except (ValueError, rasterio.errors.RasterioError) as exc:
            failed += 1
            print(f"[FAIL] {src_path}: {exc}")
            continue
        written += 1
        print(f"[OK] {src_path} -> {dst_path}")

    print(f"[DONE] processed={len(raster_paths)} written={written} skipped={skipped} failed={failed} output={output_dir}")


## Run the pipeline

This is the main execution cell. It runs the step functions in the correct order and writes a step-status CSV after each step.

Run flags:

- `RUN_CORE_RASTER_STEPS`: runs steps `01` through `05B`.
- `RUN_VECTOR_DOWNLOAD`: runs step `06A` to download or rebuild the boundary.
- `RUN_POLYGON_STATS_IF_DISTRICTS_EXIST`: runs step `06B` only when the districts file exists.
- `RUN_CLIP_TO_BOUNDARY`: runs step `07` and writes clipped webapp rasters.

For a full reproduction, keep all flags enabled. For faster debugging, you can disable later steps after their outputs already exist.

The status table is written to `outputs/summaries/notebook_full_run_step_status.csv`. If a step fails, the notebook records the failed step and then raises the original error so the problem is visible.


In [ ]:
RUN_CORE_RASTER_STEPS = True
RUN_VECTOR_DOWNLOAD = True
RUN_POLYGON_STATS_IF_DISTRICTS_EXIST = True
RUN_CLIP_TO_BOUNDARY = True

STEP_RESULTS = []


def write_step_status(cfg: PipelineConfig):
    if STEP_RESULTS:
        pd.DataFrame(STEP_RESULTS).to_csv(
            cfg.summaries_dir / "notebook_full_run_step_status.csv",
            index=False,
        )


def run_checked_step(name: str, fn):
    print(f"\n===== RUN {name} =====")
    started = time.perf_counter()
    try:
        result = fn()
    except Exception as exc:
        elapsed = round(time.perf_counter() - started, 2)
        STEP_RESULTS.append({"step": name, "status": "failed", "seconds": elapsed, "error": repr(exc)})
        write_step_status(CFG)
        print(f"===== FAILED {name} ({elapsed:.2f}s) =====")
        raise
    elapsed = round(time.perf_counter() - started, 2)
    STEP_RESULTS.append({"step": name, "status": "ok", "seconds": elapsed})
    write_step_status(CFG)
    print(f"===== OK {name} ({elapsed:.2f}s) =====")
    return result


if RUN_CORE_RASTER_STEPS:
    run_checked_step("01_split_landsat_stack", lambda: step01_split_landsat_stack(CFG))
    run_checked_step("02_temporal_baseline_lst", lambda: step02_temporal_baseline_lst(CFG))
    run_checked_step("03_temporal_hotspots", lambda: step03_temporal_hotspots(CFG))
    run_checked_step("04a_spatial_zscore", lambda: step04a_spatial_zscore(CFG))
    run_checked_step("04b_normalize", lambda: step04b_normalize(CFG))
    run_checked_step("04c_composite_indices", lambda: step04c_composite_indices(CFG))
    run_checked_step("04d_hotspot_types", lambda: step04d_hotspot_types(CFG))
    run_checked_step("04e_composite_allyears", lambda: step04e_composite_allyears(CFG))
    run_checked_step("04f_albedo_deltalst_graph", lambda: step04f_albedo_deltalst_graph(CFG))
    run_checked_step("05_prepare_webapp_rasters", lambda: step05_prepare_webapp_rasters(CFG))
    run_checked_step("05b_albedo_ndvi_delta_pairs", lambda: step05b_albedo_ndvi_delta_pairs(CFG))

if RUN_VECTOR_DOWNLOAD:
    run_checked_step("06a_download_vectors_or_build_boundary", lambda: step06a_download_vectors(CFG))

if RUN_POLYGON_STATS_IF_DISTRICTS_EXIST:
    if CFG.districts_file.exists():
        run_checked_step("06b_polygon_stats", lambda: step06b_polygon_stats(CFG))
    else:
        print(f"[SKIP] Polygon stats: districts file not found at {CFG.districts_file}")

if RUN_CLIP_TO_BOUNDARY:
    run_checked_step("07_clip_to_boundary", lambda: step07_clip_to_boundary(CFG, input_dir=CFG.webapp_rasters_dir, overwrite=True))

pd.DataFrame(STEP_RESULTS)


## Final checks

These helper functions summarize what the pipeline produced.

They inspect:

- the webapp raster manifest;
- every GeoTIFF in `outputs/webapp_rasters`;
- summary files in `outputs/summaries`;
- vector files in `outputs/webapp_vectors`.

The tables are meant for human review. They help confirm that files exist, rasters are readable, CRS/nodata metadata are present, and outputs have plausible sizes.


In [ ]:
def summarize_raster_folder(folder: Path) -> pd.DataFrame:
    rows = []
    for path in sorted(folder.glob("*.tif")):
        try:
            with rasterio.open(path) as src:
                rows.append({
                    "file": path.name,
                    "width": src.width,
                    "height": src.height,
                    "crs": str(src.crs),
                    "nodata": src.nodata,
                    "size_mb": round(path.stat().st_size / (1024 * 1024), 3),
                })
        except rasterio.errors.RasterioError as exc:
            rows.append({"file": path.name, "error": str(exc)})
    return pd.DataFrame(rows)


def summarize_outputs(cfg: PipelineConfig) -> dict[str, pd.DataFrame]:
    manifest_path = cfg.metadata_dir / "webapp_raster_manifest.json"
    if manifest_path.exists():
        manifest = pd.DataFrame(json.loads(manifest_path.read_text(encoding="utf-8")))
        if not manifest.empty and "filename" in manifest:
            manifest["exists"] = manifest["filename"].map(lambda name: (cfg.webapp_rasters_dir / name).exists())
    else:
        manifest = pd.DataFrame()

    summaries = pd.DataFrame([
        {"file": path.name, "size_kb": round(path.stat().st_size / 1024, 1)}
        for path in sorted(cfg.summaries_dir.glob("*"))
        if path.is_file()
    ])

    vectors = pd.DataFrame([
        {"file": path.name, "size_kb": round(path.stat().st_size / 1024, 1)}
        for path in sorted(cfg.webapp_vectors_dir.glob("*"))
        if path.is_file()
    ])

    return {
        "manifest": manifest,
        "webapp_rasters": summarize_raster_folder(cfg.webapp_rasters_dir),
        "summaries": summaries,
        "webapp_vectors": vectors,
    }


output_inventory = summarize_outputs(CFG)
output_inventory["manifest"]


## Validation checklist

This cell converts the final output inventory into pass/fail checks.

It verifies that:

- the expected yearly LST files exist;
- the raster manifest lists all expected webapp layers;
- all webapp rasters exist and contain valid pixels;
- expected summary CSVs exist;
- the 1 km relationship tables are not empty;
- vector outputs exist;
- clipped raster outputs exist when clipping is enabled.

The Earth Engine metadata CSV is reported separately because it is created by the remote Earth Engine export script and is not consumed by the local processing steps. A missing metadata CSV is useful to know, but it does not mean the local pipeline failed.

The validation table is written to `outputs/summaries/notebook_full_run_validation.csv`.


In [ ]:
def validate_pipeline_outputs(cfg: PipelineConfig) -> pd.DataFrame:
    period = cfg.period_label
    y = cfg.webapp_year
    expected_webapp = [
        f"LST_{y}_summer_30m.tif",
        f"anomaly_{y}_summer_30m.tif",
        f"zscore_temporal_{y}_summer_30m.tif",
        f"zscore_spatial_{y}_summer_30m.tif",
        f"NDVI_{y}_summer_30m.tif",
        f"Albedo_{y}_summer_30m.tif",
        f"DeltaLST_{y}_summer_1km.tif",
        f"HVI_{y}_summer_30m.tif",
        f"HRI_{y}_summer_30m.tif",
        f"UHEI_{y}_summer_30m.tif",
        f"hotspot_temporal_{y}_summer_30m.tif",
        f"hotspot_temporal_persistence_{period}.tif",
        f"hotspot_structural_persistence_{period}.tif",
        f"hotspot_structural_vs_anomalous_{y}.tif",
        f"climatology_mean_{period}_30m.tif",
    ]
    expected_summaries = [
        "LST_yearly_input_summary_median_30m.csv",
        "LST_summary_anomalies_median_30m.csv",
        "temporal_hotspot_summary.csv",
        "temporal_hotspot_persistence_summary.csv",
        f"zscore_spatial_{y}_summary.csv",
        f"normalized_{y}_summary.csv",
        f"composite_indices_{y}_summary.csv",
        f"hotspot_structural_vs_anomalous_new_{y}_summary.csv",
        f"hotspot_structural_vs_anomalous_{period}_yearly_counts_new.csv",
        f"hotspot_structural_vs_anomalous_persistence_{period}_summary.csv",
        f"albedo_deltalst_{y}_1km_pairs.csv",
        f"albedo_deltalst_{y}_1km_stats.csv",
        f"albedo_ndvi_delta_{y}_1km_pairs.csv",
        "notebook_full_run_step_status.csv",
    ]

    checks = []
    def add(name, ok, detail=""):
        checks.append({"check": name, "ok": bool(ok), "detail": detail})

    raw_lst_count = len(list(cfg.raw_lst_dir.glob(f"{cfg.city}_LST_*_summer_*_30m.tif")))
    add("yearly LST files", raw_lst_count == len(cfg.years_lst) * 3, f"{raw_lst_count} files")

    manifest_path = cfg.metadata_dir / "webapp_raster_manifest.json"
    manifest = json.loads(manifest_path.read_text(encoding="utf-8")) if manifest_path.exists() else []
    manifest_files = [row["filename"] for row in manifest]
    missing_manifest = sorted(set(expected_webapp) - set(manifest_files))
    add("manifest expected layers", not missing_manifest, ", ".join(missing_manifest) if missing_manifest else "all expected layers")

    raster_valid_counts = []
    missing_rasters = []
    for filename in expected_webapp:
        path = cfg.webapp_rasters_dir / filename
        if not path.exists():
            missing_rasters.append(filename)
            continue
        with rasterio.open(path) as src:
            arr = src.read(1)
            valid = np.isfinite(arr)
            if src.nodata is not None:
                valid &= arr != src.nodata
            raster_valid_counts.append(int(valid.sum()))
    add("webapp rasters exist", not missing_rasters, ", ".join(missing_rasters) if missing_rasters else f"{len(expected_webapp)} files")
    add("webapp rasters have valid pixels", bool(raster_valid_counts) and min(raster_valid_counts) > 0, f"min_valid={min(raster_valid_counts) if raster_valid_counts else 0}")

    missing_summaries = [name for name in expected_summaries if not (cfg.summaries_dir / name).exists()]
    add("summary CSVs exist", not missing_summaries, ", ".join(missing_summaries) if missing_summaries else f"{len(expected_summaries)} summaries")

    pairs_path = cfg.summaries_dir / f"albedo_deltalst_{y}_1km_pairs.csv"
    triples_path = cfg.summaries_dir / f"albedo_ndvi_delta_{y}_1km_pairs.csv"
    pairs_rows = len(pd.read_csv(pairs_path)) if pairs_path.exists() else 0
    triples_rows = len(pd.read_csv(triples_path)) if triples_path.exists() else 0
    add("albedo/deltalst pairs nonempty", pairs_rows > 0, f"{pairs_rows} rows")
    add("albedo/ndvi/deltalst pairs nonempty", triples_rows > 0, f"{triples_rows} rows")

    vector_paths = [
        cfg.boundary_file,
        cfg.districts_file,
        cfg.webapp_vectors_dir / f"districts_enriched_{y}.gpkg",
        cfg.webapp_vectors_dir / f"districts_enriched_{y}.geojson",
    ]
    missing_vectors = [str(path) for path in vector_paths if not path.exists()]
    add("vector outputs exist", not missing_vectors, ", ".join(missing_vectors) if missing_vectors else f"{len(vector_paths)} files")

    clipped_dir = cfg.output_dir / f"{cfg.webapp_rasters_dir.name}_{cfg.city_slug}_clipped"
    clipped = sorted(clipped_dir.glob("*.tif"))
    add("clipped rasters count", len(clipped) == len(expected_webapp), f"{len(clipped)} files")

    metadata_path = cfg.csv_data_download_dir / cfg.lst_metadata_file
    add("GEE metadata CSV present", metadata_path.exists(), str(metadata_path))

    out = pd.DataFrame(checks)
    out.to_csv(cfg.summaries_dir / "notebook_full_run_validation.csv", index=False)
    blocking = out[out["check"] != "GEE metadata CSV present"]
    if not blocking["ok"].all():
        raise AssertionError("One or more local pipeline validation checks failed.")
    return out

validation_results = validate_pipeline_outputs(CFG)
validation_results


In [ ]:
output_inventory["webapp_rasters"]


In [ ]:
output_inventory["summaries"].head(50)
